In [1]:
from astropy import units as u
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from astropy.table import Table
from astropy.io import fits
from astropy.visualization import simple_norm
from astropy.nddata import Cutout2D
from astropy.wcs import WCS
import re
from astropy.coordinates import SkyCoord
import sys
sys.path.append('/home/t.yoo/Paths')
import Paths.Paths as paths
import matplotlib as mpl
import matplotlib.patches as patches
Path = paths.filepaths()
plt.rcParams['axes.labelsize']=20
plt.rcParams['xtick.labelsize']=15
plt.rcParams['ytick.labelsize']=15
image_filenames ={
    "f140m": "/orange/adamginsburg/jwst/w51/F140M/pipeline/jw06151-o001_t001_nircam_clear-f140m-merged_i2d.fits",
    "f162m": "/orange/adamginsburg/jwst/w51/F162M/pipeline/jw06151-o001_t001_nircam_clear-f162m-merged_i2d.fits",
    "f182m": "/orange/adamginsburg/jwst/w51/F182M/pipeline/jw06151-o001_t001_nircam_clear-f182m-merged_i2d.fits",
    "f187n": "/orange/adamginsburg/jwst/w51/F187N/pipeline/jw06151-o001_t001_nircam_clear-f187n-merged_i2d.fits",
    "f210m": "/orange/adamginsburg/jwst/w51/F210M/pipeline/jw06151-o001_t001_nircam_clear-f210m-merged_i2d.fits",
    "f335m": "/orange/adamginsburg/jwst/w51/F335M/pipeline/jw06151-o001_t001_nircam_clear-f335m-merged_i2d.fits",
    "f360m": "/orange/adamginsburg/jwst/w51/F360M/pipeline/jw06151-o001_t001_nircam_clear-f360m-merged_i2d.fits",
    "f405n": "/orange/adamginsburg/jwst/w51/F405N/pipeline/jw06151-o001_t001_nircam_clear-f405n-merged_i2d.fits",
    "f410m": "/orange/adamginsburg/jwst/w51/F410M/pipeline/jw06151-o001_t001_nircam_clear-f410m-merged_i2d.fits", # weird, the filename is different from what is downloaded with the STScI pipeline...
    "f480m": "/orange/adamginsburg/jwst/w51/F480M/pipeline/jw06151-o001_t001_nircam_clear-f480m-merged_i2d.fits",
    "f560w": "/orange/adamginsburg/jwst/w51/F560W/pipeline/jw06151-o002_t001_miri_f560w_i2d.fits",
    "f770w": "/orange/adamginsburg/jwst/w51/F770W/pipeline/jw06151-o002_t001_miri_f770w_i2d.fits",
    "f1000w": "/orange/adamginsburg/jwst/w51/F1000W/pipeline/jw06151-o002_t001_miri_f1000w_i2d.fits",
    "f1280w": "/orange/adamginsburg/jwst/w51/F1280W/pipeline/jw06151-o002_t001_miri_f1280w_i2d.fits",
    "f1500w": "/orange/adamginsburg/jwst/w51/F1500W/pipeline/jw06151-o002_t001_miri_f1500w_i2d.fits",
    "f2100w": "/orange/adamginsburg/jwst/w51/F2100W/pipeline/jw06151-o002_t001_miri_f2100w_i2d.fits",
    "w51e_1.3mm": Path.w51e_b6_tt0,
    "w51e_3mm": Path.w51e_b3_tt0,
    "w51n_1.3mm": Path.w51n_b6_tt0,
    "w51n_3mm": Path.w51n_b3_tt0,
    "vla_22GHz": "/orange/adamginsburg/w51/TaehwaYoo/vla/2016paper/W51-K-B.S1-ICLN.DAVID-MEH.fits",
    "vla_14GHz": "/orange/adamginsburg/w51/TaehwaYoo/vla/2016paper/W51Ku_C_Aarray_continuum_2048_high_uniform.clean.image.fits",
    "vla_8GHz": "/orange/adamginsburg/w51/TaehwaYoo/vla/2016paper/W51-X-ABCD-S1.VTESS.VTC.DAVID-MEH.fits",
    "vla_5GHz": "/orange/adamginsburg/w51/TaehwaYoo/vla/2016paper/W51-CBAND-feathered.fits"
}
catalogs_filters = {"f140m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f140m_nrca_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f162m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f162m_nrca_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f182m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f182m_nrca_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f187n_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f187n_nrca_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f210m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f210m_nrca_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f335m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f335m_nrca_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f360m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f360m_nrca_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f405n_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f405n_nrca_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f410m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f410m_nrca_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f480m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f480m_nrca_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f140m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f140m_nrcb_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f162m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f162m_nrcb_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                  "f182m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f182m_nrcb_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f187n_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f187n_nrcb_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f210m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f210m_nrcb_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f335m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f335m_nrcb_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f360m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f360m_nrcb_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f405n_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f405n_nrcb_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f410m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f410m_nrcb_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f480m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f480m_nrcb_indivexp_merged_dao_after_merger_combined_with_satstars.fits',
                   "f560w": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f560w_mirimage_indivexp_merged_dao_after_merger_combined_with_satstars_fixed.fits',
                   "f770w": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f770w_mirimage_indivexp_merged_dao_after_merger_combined_with_satstars_fixed.fits',
                   "f1000w": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f1000w_mirimage_indivexp_merged_dao_after_merger_combined_with_satstars_fixed.fits',
                   "f1280w": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f1280w_mirimage_indivexp_merged_dao_after_merger_combined_with_satstars_fixed.fits',
                   "f2100w": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f2100w_mirimage_indivexp_merged_dao_after_merger_combined_with_satstars_fixed.fits',
                   "vla": '/home/t.yoo/w51/w51_nircam/analysis/vla.fits',
                  }
reprojected_dir = '/orange/adamginsburg/jwst/w51/reproject_to_alma/'



catalog = Table.read('/orange/adamginsburg/jwst/w51/catalogs/final_nircam_miri_indivexp_merged_dao_refined_after_sat.fits')
catalog = Table.read('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/final_catalog.fits')
def plot_SED(image_filenames, row_jwst, row_alma, row_vla, label, cutout_size=2*u.arcsec, alma_region='w51e'):
    fig = plt.figure(figsize=(24, 20))
    gs = GridSpec(5,6, figure=fig, wspace=0, hspace=0)
    ax_f140m = fig.add_subplot(gs[0,0])
    ax_f162m = fig.add_subplot(gs[0,1])
    ax_f182m = fig.add_subplot(gs[0,2])
    ax_f187n = fig.add_subplot(gs[0,3])
    ax_f210m = fig.add_subplot(gs[0,4])
    ax_f335m = fig.add_subplot(gs[0,5])
    ax_f360m = fig.add_subplot(gs[1,0])
    ax_f405n = fig.add_subplot(gs[1,1])
    ax_f410m = fig.add_subplot(gs[1,2])
    ax_f480m = fig.add_subplot(gs[1,3])
    ax_f560w = fig.add_subplot(gs[1,4])
    ax_f770w = fig.add_subplot(gs[1,5])
    ax_f1000w = fig.add_subplot(gs[2,0])
    ax_f1280w = fig.add_subplot(gs[2,1])
    ax_f2100w = fig.add_subplot(gs[2,2])
    ax_b6 = fig.add_subplot(gs[2,3])
    ax_b3 = fig.add_subplot(gs[2,4])
    ax_vla_5GHz = fig.add_subplot(gs[2,5])
    ax_vla_8GHz = fig.add_subplot(gs[3,0])
    ax_vla_14GHz = fig.add_subplot(gs[3,1])
    ax_vla_22GHz = fig.add_subplot(gs[3,2])
    ax_images = [ax_f140m, ax_f162m, ax_f182m, ax_f187n, ax_f210m, ax_f335m, ax_f360m, ax_f405n,
                 ax_f410m, ax_f480m, ax_f560w, ax_f770w, ax_f1000w, ax_f1280w, ax_f2100w, ax_b6, ax_b3, ax_vla_5GHz, ax_vla_8GHz, ax_vla_14GHz, ax_vla_22GHz]
    ax_main = fig.add_subplot(gs[4, :])
    filter_names = ["f140m", "f162m", "f182m", "f187n", "f210m", "f335m", "f360m", "f405n",
                    "f410m", "f480m", "f560w", "f770w", "f1000w", "f1280w", "f2100w", "1.3mm", "3mm", "vla_5GHz", "vla_8GHz", "vla_14GHz", "vla_22GHz"]
    skycoords = SkyCoord(ra=catalog['skycoord_ra'][idx]*u.deg, dec=catalog['skycoord_dec'][idx]*u.deg)
    print('skycoords in h:m:s:', skycoords.to_string('hmsdms'))
    print(len(filter_names), len(ax_images))
    for i, ax in enumerate(ax_images):  
        img_b3 = image_filenames[f'{alma_region}_3mm']
        header_b3 = fits.open(img_b3)[0].header
        pixel_scale_b3 = WCS(header_b3, naxis=2).proj_plane_pixel_scales()[0]
        img_b6 = image_filenames[f'{alma_region}_1.3mm']
        header_b6 = fits.open(img_b6)[0].header
        pixel_scale_b6 = WCS(header_b6, naxis=2).proj_plane_pixel_scales()[0]

        if filter_names[i] in ["1.3mm", "3mm"]: 
            if filter_names[i] == "1.3mm":
                band = 'b6'
            elif filter_names[i] == "3mm":
                band = 'b3'
            img_filename = image_filenames[f"{alma_region}_{filter_names[i]}"]
            print('alma image filename:', img_filename)
            img = fits.open(img_filename)[0].data[0][0]
            header = fits.open(img_filename)[0].header
            wcs = WCS(header, naxis=2)
        elif filter_names[i] in ['f140m', 'f162m', 'f182m', 'f187n', 'f210m', 'f335m', 'f360m', 'f405n', 'f410m', 'f480m', 'f560w', 'f770w', 'f1000w', 'f1280w', 'f2100w']:
            #filt}_reprojected_to_alma_w51n_b6.fits
            img_filename = reprojected_dir + f"{filter_names[i]}_reprojected_to_alma_{alma_region}_b3.fits"
            print('JWST image filename:', img_filename)
            img = fits.open(img_filename)[0].data
            header = fits.open(img_filename)[0].header
            wcs = WCS(header, naxis=2)
        else:
            
            image_filename = reprojected_dir + f"{filter_names[i]}_reprojected_to_alma_{alma_region}_b3.fits"
            img = fits.open(image_filename)[0].data
            if not len(img.shape) == 2:
                img = img[0][0]
            header = fits.open(image_filename)[0].header
            wcs = WCS(header, naxis=2)
          

        try:
        
            cutout = Cutout2D(img, skycoords, (cutout_size, cutout_size), wcs=wcs)
            norm = simple_norm(cutout.data, 'sqrt', percent=99.5)
            ax.imshow(cutout.data, norm=norm, origin='lower', cmap='inferno')
            if filter_names[i] == "1.3mm":
                pixel_scale = pixel_scale_b6
            else:
                pixel_scale = pixel_scale_b3
            print('pixel_scale:', pixel_scale)
                
            circle = patches.Circle((cutout.data.shape[1]/2, cutout.data.shape[0]/2), radius=(0.1*u.arcsec/pixel_scale).to(u.deg/u.deg).value, edgecolor='cyan', facecolor='none', lw=2)
            ax.add_patch(circle)
            print(filter_names[i])
            ax.text(0.1, 0.9, filter_names[i].upper(), transform=ax.transAxes, fontsize=12, bbox=dict(facecolor='white', alpha=0.7))
            ax.axis('off')
            ax.set_xlim(0, cutout.data.shape[1])
            ax.set_ylim(0, cutout.data.shape[0])
        except Exception as e:
            print(img.shape)
            pixcoord = skycoords.to_pixel(wcs)
            print('pixcoord:', pixcoord)
            print(f"Could not create cutout for filter {filter_names[i]}: {e}")
            continue      
        
            
        if not filter_names[i] in ['1.3mm', '3mm']:
            if filter_names[i] in ['f140m', 'f162m', 'f182m', 'f187n', 'f210m', 'f335m', 'f360m', 'f405n', 'f410m', 'f480m']:
                cat_nrca = catalogs_filters[f'{filter_names[i]}_nrca']
                skycoord_nrca = Table.read(cat_nrca)['skycoord']
                print('catalog filename cat_nrca:', cat_nrca)
                pixcoord_nrca = skycoord_nrca.to_pixel(cutout.wcs)
                cat_nrcb = catalogs_filters[f'{filter_names[i]}_nrcb']
                print('catalog filename cat_nrcb:', cat_nrcb)

                skycoord_nrcb = Table.read(cat_nrcb)['skycoord']
                pixcoord_nrcb = skycoord_nrcb.to_pixel(cutout.wcs)
                ax.scatter(pixcoord_nrca[0], pixcoord_nrca[1], facecolor='none', color='blue', s=10)
                ax.scatter(pixcoord_nrcb[0], pixcoord_nrcb[1], facecolor='none', color='red', s=10)
                
            
            elif filter_names[i] in ['f560w', 'f770w', 'f1000w', 'f1280w', 'f2100w']:
                cat_miri = catalogs_filters[f'{filter_names[i]}']
                print('catalog filename cat_miri:', cat_miri)
                skycoord_miri = Table.read(cat_miri)['skycoord']
                pixcoord_miri = skycoord_miri.to_pixel(cutout.wcs)
                ax.scatter(pixcoord_miri[0], pixcoord_miri[1], facecolor='none', color='green', s=10)

            else:
                
                cat_vla = Table.read(catalogs_filters['vla'])
                print('catalog filename cat_vla:', catalogs_filters['vla'])
                ra = cat_vla['GRAdeg']
                dec = cat_vla['GDEdeg']
                skycoord_vla = SkyCoord(ra=ra, dec=dec)
                pixcoord_vla = skycoord_vla.to_pixel(cutout.wcs)
                ax.scatter(pixcoord_vla[0], pixcoord_vla[1], facecolor='none', color='magenta', s=10)

        # limit xlim ylim as same as cutout size
       
    # plot SED
    colors = mpl.cm.viridis(np.linspace(0, 1, len(filter_names)))
    fluxarr = []
    for i, filter_name in enumerate(filter_names):
        # Get the effective wavelength for each filter
      

        if filter_name == '1.3mm':
            wav = 130000
            flux = row_alma['flux_b6']
        elif filter_name == '3mm':
            wav = 300000
            flux = row_alma['flux_b3']
        elif filter_name in ["vla_5GHz", "vla_8GHz", "vla_14GHz", "vla_22GHz"]:
            if row_vla is not None:
                freq = row_vla['freq']
                wav = (3e8 / (freq * 1e9)) * 1e6 # convert frequency in GHz to wavelength in micron
                flux = row_vla['flux']
        else:
            print(filter_name)
            
            wav = int(filter_name[1:-1])
            flux = row_jwst['flux_fit_' + filter_name]
        

        marker='o'

        if np.isnan(flux) or flux <= 0  or np.ma.is_masked(flux) or filter_names[i] not in ['1.3mm', '3mm']: # or flux array is masked
            if filter_names[i] in ['f140m', 'f162m', 'f182m', 'f187n', 'f210m', 'f335m', 'f360m', 'f405n', 'f410m', 'f480m']:
                print(f"Flux for filter {filter_name} is NaN or non-positive, checking catalogs for upper limits...")
                cat_nrca = catalogs_filters[f'{filter_names[i]}_nrca']
                skycoord_nrca = Table.read(cat_nrca)['skycoord']
                cat_nrcb = catalogs_filters[f'{filter_names[i]}_nrcb']
                skycoord_nrcb = Table.read(cat_nrcb)['skycoord']
                # get the sources that are within 0.1 arcsec from the target source in the catalog, and use their fluxes as upper limits
                idx_nrca = skycoord_nrca.separation(skycoords) < 0.1*u.arcsec
                idx_nrcb = skycoord_nrcb.separation(skycoords) < 0.1*u.arcsec
                if np.any(idx_nrca):
                    flux_nrca = Table.read(cat_nrca)['flux_fit'][idx_nrca]
                    flux_nrca = flux_nrca[~np.isnan(flux_nrca)]
                    if len(flux_nrca) > 0:
                        flux = np.max(flux_nrca)
                if np.any(idx_nrcb):
                    flux_nrcb = Table.read(cat_nrcb)['flux_fit'][idx_nrcb]
                    flux_nrcb = flux_nrcb[~np.isnan(flux_nrcb)]
                    if len(flux_nrcb) > 0:
                        flux = np.max(flux_nrcb)
            elif filter_names[i] in ['f560w', 'f770w', 'f1000w', 'f1280w', 'f2100w']:
                cat_miri = catalogs_filters[f'{filter_names[i]}']
                skycoord_miri = Table.read(cat_miri)['skycoord']
                idx_miri = skycoord_miri.separation(skycoords) < 0.1*u.arcsec
                if np.any(idx_miri):
                    flux_miri = Table.read(cat_miri)['flux_fit'][idx_miri]
                    flux_miri = flux_miri[~np.isnan(flux_miri)]
                    if len(flux_miri) > 0:
                        flux = np.max(flux_miri)


                    
            marker = 'x'
                

        print(f"Filter: {filter_name}, Wavelength: {wav} micron, Flux: {flux} Jy")
        ax_main.plot(wav / 100.0, flux, color = colors[i], marker=marker, markersize=20, label=filter_name.upper())
        ax_main.vlines(wav / 100.0, ymin=1e-10, ymax=1e10, colors=colors[i], linestyles='dashed', alpha=0.5)
        fluxarr.append(flux)
    fluxarr = np.array(fluxarr)
    ax_main.set_xscale('log')
    ax_main.set_yscale('log')
    ax_main.set_xlabel('Wavelength (micron)')
    ax_main.set_ylabel('Flux (Jy)')
    ax_main.text(0.7, 0.9, f'SED for Source {label}', transform=ax_main.transAxes, fontsize=26)
    #ax_main.legend(fontsize=12, ncol=4, bbox_to_anchor=(0.55, 0, 0.2,0.4))
    ax_main.set_ylim(np.nanmin(fluxarr)*0.5, np.nanmax(fluxarr)*100)
    ax_main.set_xlim(1, 5000000)
    plt.tight_layout()
    plt.savefig(f'/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/plots/seds/{alma_region}_source_{label}_SED.png')
    plt.close()



def get_number_after_hash(name):
    match = re.search(r'#(\d+)', name)
    if match:
        return int(match.group(1))
    else:
        return None 
        
def extract_sort_key(name):
    # Extract prefix before #
    prefix_match = re.match(r'(.*)#(\d+)', name)
    if prefix_match:
        prefix = prefix_match.group(1)
        number = int(prefix_match.group(2))
        return (prefix, number)
    else:
        return (name, -1)  # fallback if no match
"""
sheet_id = '1FRTQynXdrCuc-uwGIOKizEnDhFVoe9d7IuZ0CFuWIJ8'
sheet_name = 'Sheet1'
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid=0"
tb = Table.read(url, format='ascii.csv')
region_name = tb['region_name']
alma_band = tb['alma_band']
jwst_filt = tb['jwst_filter']
alma_overlap_list = [str(name) for name in region_name if str(name).startswith('alma_overlap')]
alma_overlap_list_sorted = sorted(alma_overlap_list, key=extract_sort_key)

sort_keys = [extract_sort_key(name) for name in alma_overlap_list]
sorting_index = sorted(range(len(alma_overlap_list)), key=lambda i: sort_keys[i])
alma_band_overlap = [str(alma_band[region_name == name][0]) for name in alma_overlap_list]
jwst_filt_overlap = [str(jwst_filt[region_name == name][0]) for name in alma_overlap_list]
print(sorting_index)
alma_band_sorted = [alma_band_overlap[i] for i in sorting_index]
jwst_filt_sorted = [jwst_filt_overlap[i] for i in sorting_index]


print(alma_overlap_list_sorted)
print(alma_band_sorted)
print(jwst_filt_sorted)

print(len(alma_overlap_list_sorted))
print(len(alma_band_sorted))
print(len(jwst_filt_sorted))
"""



'\nsheet_id = \'1FRTQynXdrCuc-uwGIOKizEnDhFVoe9d7IuZ0CFuWIJ8\'\nsheet_name = \'Sheet1\'\nurl = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid=0"\ntb = Table.read(url, format=\'ascii.csv\')\nregion_name = tb[\'region_name\']\nalma_band = tb[\'alma_band\']\njwst_filt = tb[\'jwst_filter\']\nalma_overlap_list = [str(name) for name in region_name if str(name).startswith(\'alma_overlap\')]\nalma_overlap_list_sorted = sorted(alma_overlap_list, key=extract_sort_key)\n\nsort_keys = [extract_sort_key(name) for name in alma_overlap_list]\nsorting_index = sorted(range(len(alma_overlap_list)), key=lambda i: sort_keys[i])\nalma_band_overlap = [str(alma_band[region_name == name][0]) for name in alma_overlap_list]\njwst_filt_overlap = [str(jwst_filt[region_name == name][0]) for name in alma_overlap_list]\nprint(sorting_index)\nalma_band_sorted = [alma_band_overlap[i] for i in sorting_index]\njwst_filt_sorted = [jwst_filt_overlap[i] for i in sorting_index]\n\n\nprint(alma_ove

In [2]:
w51e_alma_catalog = '/blue/adamginsburg/t.yoo/from_red/w51/w51_frag_new/dendro/tables/dendro_w51e_master.fits'

w51n_alma_catalog = '/blue/adamginsburg/t.yoo/from_red/w51/w51_frag_new/dendro/tables/dendro_w51n_master.fits'

w51e_matched_idx = [3, 7, 36, 66, 67, 68, 76, 78, 93, 95, 106]
w51n_matched_idx = [19, 22, 29, 30, 58, 59, 64, 66, 76, 89, 90, 91]

w51e_alma_ra = Table.read(w51e_alma_catalog)['ra']
w51e_alma_dec = Table.read(w51e_alma_catalog)['dec']

vla_cat_file = './vla_updated.fits'
vla_cat = Table.read(vla_cat_file)
jwst_complete_catalog = '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/final_catalog.fits'
jwst_tab = Table.read(jwst_complete_catalog)
vla_cat.pprint(max_width=-1, max_lines=-1)

# if vla_cat has duplicated value in freq, leave only the one with the highest recno
freqs = vla_cat['Freq']
unique_freqs, counts = np.unique(freqs, return_counts=True)
duplicated_freqs = unique_freqs[counts > 1]
print("Duplicated frequencies:", duplicated_freqs)

# Keep only the rows with the highest recno for each duplicated frequency
vla_cat_filtered = vla_cat.copy()
for freq in duplicated_freqs:
    rows_with_freq = np.where(vla_cat_filtered['Freq'] == freq)[0]
    max_recno = np.max(vla_cat_filtered['recno'][rows_with_freq])
    rows_to_remove = rows_with_freq[vla_cat_filtered['recno'][rows_with_freq] != max_recno]
    vla_cat_filtered.remove_rows(rows_to_remove)

# Use the filtered catalog
vla_cat = vla_cat_filtered

# use Bmaj and Bmin to calculate the beam size and use it to convert flux in Jy/beam to Jy
vla_cat['beam_size'] = np.sqrt(vla_cat['Bmaj'] * vla_cat['Bmin'])
vla_cat_updated = vla_cat_filtered.copy()
vla_cat_updated['FluxAp_jy'] = vla_cat['FluxAp'] * (vla_cat['beam_size'] / (3600 * 180 / np.pi))**2

print(f"Original catalog had {len(Table.read(vla_cat_file))} rows, filtered catalog has {len(vla_cat)} rows")

   Name         Label                Obs_date          Freq  Ep       Bmaj            Bmin             FluxAp              FluxPk            rms          GRAdeg        GDEdeg     recno  
                                                       GHz           arcsec          arcsec          Jy / beam           Jy / beam        Jy / beam        deg           deg              
---------- ---------------- -------------------------- ---- --- --------------- --------------- ------------------- ------------------- ------------- ------------- ------------- --------
        d2 2.5 GHz  Epoch 2 2012-10-16T02:26:38.499999  2.5   2  0.543913006783  0.526558637619  2.620970602350e-03  5.482930224390e-03  2.083390e-04 290.915843188 14.5187235922        1
        d2 3.5 GHz  Epoch 2 2012-10-16T02:26:38.499999  3.5   2  0.409057587385  0.388352155685  1.761933618500e-03  4.366006702180e-03  6.324020e-05 290.915967686 14.5183285904        2
        d2 4.9 GHz  Epoch 1 1992-10-25T00:00:00.000000  4.9   1  

In [3]:

for ii,id in enumerate(w51e_matched_idx):
    if True:
        skycoord_alma = SkyCoord(ra=w51e_alma_ra[id]*u.deg, dec=w51e_alma_dec[id]*u.deg)
        skycoord_jwst = SkyCoord(ra=jwst_tab['skycoord_ra'], dec=jwst_tab['skycoord_dec'])
        idx, d2d, d3d = skycoord_alma.match_to_catalog_sky(skycoord_jwst)
        print(f"W51e ALMA source index: {id}, matched JWST catalog index: {idx}, separation: {d2d.arcsec} arcsec")
        row_jwst = jwst_tab[idx]
        row_alma = Table.read(w51e_alma_catalog)[id]
        vla_ra = vla_cat_updated['GRAdeg']
        vla_dec = vla_cat_updated['GDEdeg']
        finite_idx = np.isfinite(vla_ra) & np.isfinite(vla_dec)
        vla_cat_updated = vla_cat_updated[finite_idx]
        vla_ra= vla_cat_updated['GRAdeg']
        vla_dec = vla_cat_updated['GDEdeg']
        
        skycoord_vla = SkyCoord(ra=vla_ra, dec=vla_dec)
        # get the indices of the VLA sources that are within 0.1 arcsec from the ALMA source
        separations = skycoord_vla.separation(skycoord_alma)
        vla_match_idx = np.where(separations < 0.1*u.arcsec)[0]
        if len(vla_match_idx) > 0:
            row_vla = vla_cat_updated[vla_match_idx]
            print(f"Found {len(row_vla)} VLA sources within 0.1 arcsec of ALMA source {id}")
        else:
            row_vla = None
        plot_SED(image_filenames, row_jwst, row_alma, row_vla, f"W51E_{id}")

W51e ALMA source index: 3, matched JWST catalog index: 23919, separation: [0.03169011] arcsec
skycoords in h:m:s: 19h23m43.79054414s +14d30m19.79193444s
21 21


Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


JWST image filename: /orange/adamginsburg/jwst/w51/reproject_to_alma/f140m_reprojected_to_alma_w51e_b3.fits
pixel_scale: 1.944444444444e-06 deg
f140m
catalog filename cat_nrca: /orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f140m_nrca_indivexp_merged_dao_after_merger_combined_with_satstars.fits
catalog filename cat_nrcb: /orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f140m_nrcb_indivexp_merged_dao_after_merger_combined_with_satstars.fits


Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


JWST image filename: /orange/adamginsburg/jwst/w51/reproject_to_alma/f162m_reprojected_to_alma_w51e_b3.fits
pixel_scale: 1.944444444444e-06 deg
f162m
catalog filename cat_nrca: /orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f162m_nrca_indivexp_merged_dao_after_merger_combined_with_satstars.fits
catalog filename cat_nrcb: /orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f162m_nrcb_indivexp_merged_dao_after_merger_combined_with_satstars.fits


...

Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


pixel_scale: 1.944444444444e-06 deg
vla_14GHz
catalog filename cat_vla: /home/t.yoo/w51/w51_nircam/analysis/vla.fits


Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


pixel_scale: 1.944444444444e-06 deg
vla_22GHz
catalog filename cat_vla: /home/t.yoo/w51/w51_nircam/analysis/vla.fits
f140m
Flux for filter f140m is NaN or non-positive, checking catalogs for upper limits...
Filter: f140m, Wavelength: 140 micron, Flux: 11.777714680363339 Jy
f162m
Flux for filter f162m is NaN or non-positive, checking catalogs for upper limits...
Filter: f162m, Wavelength: 162 micron, Flux: 255.5571735992182 Jy
f182m
Flux for filter f182m is NaN or non-positive, checking catalogs for upper limits...
Filter: f182m, Wavelength: 182 micron, Flux: 1649.3466389443695 Jy
f187n
Flux for filter f187n is NaN or non-positive, checking catalogs for upper limits...
Filter: f187n, Wavelength: 187 micron, Flux: 1990.8015237558175 Jy
f210m
Flux for filter f210m is NaN or non-positive, checking catalogs for upper limits...
Filter: f210m, Wavelength: 210 micron, Flux: 7043.526009836175 Jy
f335m
Flux for filter f335m is NaN or non-positive, checking catalogs for upper limits...
Filter: f3

/scratch/local/26430805/ipykernel_2562977/231378945.py:273: UserWarning: Warning: converting a masked element to nan.
  fluxarr = np.array(fluxarr)


In [4]:
w51e_alma_catalog = '/blue/adamginsburg/t.yoo/from_red/w51/w51_frag_new/dendro/tables/dendro_w51e_master.fits'

w51n_alma_catalog = '/blue/adamginsburg/t.yoo/from_red/w51/w51_frag_new/dendro/tables/dendro_w51n_master.fits'

w51e_matched_idx = [3, 7, 36, 66, 67, 68, 76, 78, 93, 95, 106]
w51n_matched_idx = [19, 22, 29, 30, 58, 59, 64, 66, 76, 89, 90, 91]

w51e_alma_ra = Table.read(w51e_alma_catalog)['ra']
w51e_alma_dec = Table.read(w51e_alma_catalog)['dec']

jwst_complete_catalog = '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/final_catalog.fits'
jwst_tab = Table.read(jwst_complete_catalog)

In [5]:
w51n_alma_ra = Table.read(w51n_alma_catalog)['ra']
w51n_alma_dec = Table.read(w51n_alma_catalog)['dec']

for ii,id in enumerate(w51n_matched_idx):
    if True:
        skycoord_alma = SkyCoord(ra=w51n_alma_ra[id]*u.deg, dec=w51n_alma_dec[id]*u.deg)
        skycoord_jwst = SkyCoord(ra=jwst_tab['skycoord_ra'], dec=jwst_tab['skycoord_dec'])
        idx, d2d, d3d = skycoord_alma.match_to_catalog_sky(skycoord_jwst)
        print(f"W51n ALMA source index: {id}, matched JWST catalog index: {idx}, separation: {d2d.arcsec} arcsec")
        row_jwst = jwst_tab[idx]
        row_alma = Table.read(w51n_alma_catalog)[id]
        vla_ra = vla_cat_updated['GRAdeg']
        vla_dec = vla_cat_updated['GDEdeg']
        finite_idx = np.isfinite(vla_ra) & np.isfinite(vla_dec)
        vla_cat_updated = vla_cat_updated[finite_idx]
        vla_ra= vla_cat_updated['GRAdeg']
        vla_dec = vla_cat_updated['GDEdeg']
        
        skycoord_vla = SkyCoord(ra=vla_ra, dec=vla_dec)
        # get the indices of the VLA sources that are within 0.1 arcsec from the ALMA source
        separations = skycoord_vla.separation(skycoord_alma)
        vla_match_idx = np.where(separations < 0.1*u.arcsec)[0]
        if len(vla_match_idx) > 0:
            row_vla = vla_cat_updated[vla_match_idx]
            print(f"Found {len(row_vla)} VLA sources within 0.1 arcsec of ALMA source {id}")
        else:
            row_vla = None
        plot_SED(image_filenames, row_jwst, row_alma, row_vla, f"W51_IRS2_{id}")

W51n ALMA source index: 19, matched JWST catalog index: 6784, separation: [0.04504192] arcsec
skycoords in h:m:s: 19h23m40.11657748s +14d31m05.74094345s
21 21
JWST image filename: /orange/adamginsburg/jwst/w51/reproject_to_alma/f140m_reprojected_to_alma_w51e_b3.fits
(14500, 14500)
pixcoord: (array(15119.19311382), array(11698.97812882))
Could not create cutout for filter f140m: Arrays do not overlap.


Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


JWST image filename: /orange/adamginsburg/jwst/w51/reproject_to_alma/f162m_reprojected_to_alma_w51e_b3.fits
(14500, 14500)
pixcoord: (array(15119.19311382), array(11698.97812882))
Could not create cutout for filter f162m: Arrays do not overlap.
JWST image filename: /orange/adamginsburg/jwst/w51/reproject_to_alma/f182m_reprojected_to_alma_w51e_b3.fits
(14500, 14500)
pixcoord: (array(15119.19311382), array(11698.97812882))
Could not create cutout for filter f182m: Arrays do not overlap.
JWST image filename: /orange/adamginsburg/jwst/w51/reproject_to_alma/f187n_reprojected_to_alma_w51e_b3.fits
(14500, 14500)
pixcoord: (array(15119.19311382), array(11698.97812882))
Could not create cutout for filter f187n: Arrays do not overlap.
JWST image filename: /orange/adamginsburg/jwst/w51/reproject_to_alma/f210m_reprojected_to_alma_w51e_b3.fits
(14500, 14500)
pixcoord: (array(15119.19311382), array(11698.97812882))
Could not create cutout for filter f210m: Arrays do not overlap.
JWST image filename:

Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


JWST image filename: /orange/adamginsburg/jwst/w51/reproject_to_alma/f405n_reprojected_to_alma_w51e_b3.fits
(14500, 14500)
pixcoord: (array(15119.19311382), array(11698.97812882))
Could not create cutout for filter f405n: Arrays do not overlap.
JWST image filename: /orange/adamginsburg/jwst/w51/reproject_to_alma/f410m_reprojected_to_alma_w51e_b3.fits
(14500, 14500)
pixcoord: (array(15119.19311382), array(11698.97812882))
Could not create cutout for filter f410m: Arrays do not overlap.
JWST image filename: /orange/adamginsburg/jwst/w51/reproject_to_alma/f480m_reprojected_to_alma_w51e_b3.fits
(14500, 14500)
pixcoord: (array(15119.19311382), array(11698.97812882))
Could not create cutout for filter f480m: Arrays do not overlap.
JWST image filename: /orange/adamginsburg/jwst/w51/reproject_to_alma/f560w_reprojected_to_alma_w51e_b3.fits
(14500, 14500)
pixcoord: (array(15119.19311382), array(11698.97812882))
Could not create cutout for filter f560w: Arrays do not overlap.
JWST image filename:

...

Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


JWST image filename: /orange/adamginsburg/jwst/w51/reproject_to_alma/f1280w_reprojected_to_alma_w51e_b3.fits
(14500, 14500)
pixcoord: (array(16173.2591449), array(15287.95761826))
Could not create cutout for filter f1280w: Arrays do not overlap.
JWST image filename: /orange/adamginsburg/jwst/w51/reproject_to_alma/f2100w_reprojected_to_alma_w51e_b3.fits
(14500, 14500)
pixcoord: (array(16173.2591449), array(15287.95761826))
Could not create cutout for filter f2100w: Arrays do not overlap.
alma image filename: /orange/adamginsburg/w51/TaehwaYoo/w51e_b6_imaging_2025/w51e2.spw0thru19.14500.robust0.thr0.1mJy.mfs.I.startmod.selfcal7.image.tt0.pbcor.fits
(14500, 14500)
pixcoord: (array(22865.70341647), array(21316.42584094))
Could not create cutout for filter 1.3mm: Arrays do not overlap.
alma image filename: /orange/adamginsburg/w51/TaehwaYoo/2017.1.00293.S_W51_B3_LB/may2021_successful_imaging/w51e2.spw0thru19.14500.robust0.thr0.075mJy.mfs.I.startmod.selfcal7.image.tt0.pbcor.fits
(14500, 1450

Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


(14500, 14500)
pixcoord: (array(16173.2591449), array(15287.95761826))
Could not create cutout for filter vla_14GHz: Arrays do not overlap.
(14500, 14500)
pixcoord: (array(16173.2591449), array(15287.95761826))
Could not create cutout for filter vla_22GHz: Arrays do not overlap.
f140m
Flux for filter f140m is NaN or non-positive, checking catalogs for upper limits...
Filter: f140m, Wavelength: 140 micron, Flux: 172.25513808963584 Jy
f162m
Flux for filter f162m is NaN or non-positive, checking catalogs for upper limits...
Filter: f162m, Wavelength: 162 micron, Flux: -- Jy
f182m
Flux for filter f182m is NaN or non-positive, checking catalogs for upper limits...
Filter: f182m, Wavelength: 182 micron, Flux: 6056.57305026785 Jy
f187n
Flux for filter f187n is NaN or non-positive, checking catalogs for upper limits...
Filter: f187n, Wavelength: 187 micron, Flux: 37202.31961926505 Jy
f210m
Flux for filter f210m is NaN or non-positive, checking catalogs for upper limits...
Filter: f210m, Wavele

/scratch/local/26430805/ipykernel_2562977/231378945.py:273: UserWarning: Warning: converting a masked element to nan.
  fluxarr = np.array(fluxarr)
